In [4]:
import torch
import numpy as np
import math
from concurrent.futures import ProcessPoolExecutor, as_completed
import datetime

In [12]:
# 기존 셀 1: Lost Sales 문제 전체를 아래 코드로 대체
# 셀 1: Lost Sales 문제

from dcl_definitions_final import (
    LostSalesEnv, 
    BaseStockPolicy, 
    DCLTrainer,
    collect_samples_for_thread
)

if __name__ == '__main__':
    # --- 1. 문제 및 하이퍼파라미터 정의 ---
    print("="*50)
    print("Lost Sales 문제에 대한 DCL 학습을 시작합니다.")
    print("="*50)

    # 변수 이름이 겹치지 않도록 '_ls' 접미사를 사용합니다.
    
    # <<< 수정된 부분: 이제 변수를 따로 선언하지 않습니다. >>>

    env_ls = LostSalesEnv(
        lead_time=3,
        holding_cost=1000,
        penalty_cost=5000,
        # <<< 수정된 부분: 평균과 표준편차를 직접 전달 >>>
        mean_demand=150,
        demand_std_dev=50,
        max_action=300
    )
    initial_policy_ls = BaseStockPolicy(base_stock_level=400)#초기정책(=BSP)을 400으로 설정

    # DCL 하이퍼파라미터
    n = 2     # 정책 반복 횟수
    N = 300   # 총 데이터 샘플 수, 오래걸려서 500->300으로 낮춤
    M = 10   # 상태-행동 쌍 당 외생 시나리오 수, 총 시뮬레이션 예산(Bₛ)을 결정하는 값, 오래걸려서 100에서 10으로 낮춤
    H = 20    # 롤아웃 깊이
    L = 10    # 웜업 기간, 초기값 50->10으로 낮춤
    w = 6    # 병렬 워커(프로세스) 수, w는 작업 프로세스에서 CPU 논리프로세스 수에 따라서

    # --- 2. DCL 트레이너 생성 및 학습 실행 ---
    dcl_trainer_ls = DCLTrainer(env_ls, initial_policy_ls, n, N, M, H, L, w)
    final_policies_ls = dcl_trainer_ls.train()

    # --- 3. 결과 ---
    best_policy_lost_sales = None
    if final_policies_ls:
        best_policy_lost_sales = final_policies_ls[-1]
        print("\n학습 완료! 'best_policy_lost_sales'가 생성되었습니다.")
        
        # 예시 상태: 파이프라인 재고 [5, 10, 8], 요일 벡터 [0, 0, 0, 0, 1, 0, 0] (금요일)
        example_state_ls = np.array([5, 10, 8, 0, 0, 0, 0, 1, 0, 0])
        action_ls = best_policy_lost_sales.get_action(example_state_ls)
        print(f"상태 {example_state_ls}에서 학습된 정책의 결정 행동: {action_ls}")
    else:
        print("\nLost Sales 학습이 완료되지 않았습니다.")

Lost Sales 문제에 대한 DCL 학습을 시작합니다.
Lost Sales: 자동 생성된 요일별 평균 수요:
  - 월: 158.64
  - 화: 191.03
  - 수: 164.97
  - 목: 169.64
  - 금: 62.06
  - 토: 202.69
  - 일: 200.59
DCL Trainer가 cuda 장치를 사용하여 학습을 시작합니다.

======= 시작: 근사 정책 반복 1/2 =======
데이터셋 K_0 생성을 시작합니다 (총 300개 샘플, 6개 워커)...

데이터셋 생성 완료. 총 300개 샘플 수집.
신경망 학습 시작...
Epoch 1/50, Validation Loss: 4.5802
Epoch 2/50, Validation Loss: 4.2181
Epoch 3/50, Validation Loss: 4.0242
Epoch 4/50, Validation Loss: 4.0364
Epoch 5/50, Validation Loss: 4.0437
Epoch 6/50, Validation Loss: 4.0610
Epoch 7/50, Validation Loss: 4.0903
Epoch 8/50, Validation Loss: 4.1496
조기 종료(Early stopping) 발동!
신경망 학습 완료.

======= 시작: 근사 정책 반복 2/2 =======
데이터셋 K_1 생성을 시작합니다 (총 300개 샘플, 6개 워커)...

데이터셋 생성 완료. 총 300개 샘플 수집.
신경망 학습 시작...
Epoch 1/50, Validation Loss: 7.6833
Epoch 2/50, Validation Loss: 6.4855
Epoch 3/50, Validation Loss: 6.0692
Epoch 4/50, Validation Loss: 5.8417
Epoch 5/50, Validation Loss: 5.7237
Epoch 6/50, Validation Loss: 5.7339
Epoch 7/50, Validation Loss: 5.

In [16]:
# 요일 정보: 금요일 (인덱스 4)
day_of_week_vector = np.array([0, 0, 1, 0, 0, 0, 0])

# 시나리오 1: 재고가 하나도 없는 상황 ( 모두 0)
state_1_inventory = np.array([0, 0, 0])
state_1_full = np.concatenate((state_1_inventory, day_of_week_vector))
action_1 = best_policy_lost_sales.get_action(state_1_full)
print(f"상황 1: 재고가 전혀 없을 때 (요일 포함) {state_1_full}")
print(f"▶ 추천 주문량: {action_1} 개")
print("-" * 50)


# 시나리오 2: 재고는 적지만, 곧 많이 들어올 예정인 상황
state_2_inventory = np.array([20, 100, 15])
state_2_full = np.concatenate((state_2_inventory, day_of_week_vector))
action_2 = best_policy_lost_sales.get_action(state_2_full)
print(f"상황 2: 재고는 적지만 파이프라인이 꽉 찼을 때 (요일 포함) {state_2_full}")
print(f"▶ 추천 주문량: {action_2} 개")
print("-" * 50)


# 시나리오 3: 재고가 매우 많은 상황
state_3_inventory = np.array([140, 30, 10]) # 0000은 0과 동일하므로 0으로 변경
state_3_full = np.concatenate((state_3_inventory, day_of_week_vector))
action_3 = best_policy_lost_sales.get_action(state_3_full)
print(f"상황 3: 재고가 매우 많을 때 (요일 포함) {state_3_full}")
print(f"▶ 추천 주문량: {action_3} 개")
print("-" * 50)


상황 1: 재고가 전혀 없을 때 (요일 포함) [0 0 0 0 0 1 0 0 0 0]
▶ 추천 주문량: 300 개
--------------------------------------------------
상황 2: 재고는 적지만 파이프라인이 꽉 찼을 때 (요일 포함) [ 20 100  15   0   0   1   0   0   0   0]
▶ 추천 주문량: 180 개
--------------------------------------------------
상황 3: 재고가 매우 많을 때 (요일 포함) [140  30  10   0   0   1   0   0   0   0]
▶ 추천 주문량: 163 개
--------------------------------------------------


In [22]:

from dcl_definitions_final import (
    LostSalesEnv, 
    BaseStockPolicy, 
    DCLTrainer,
    collect_samples_for_thread
)
# best_policy_lost_sales와 학습에 사용된 파라미터들이 있다고 가정
# --- 1. 저장하고 싶은 모든 파라미터를 딕셔너리로 정리 ---
# DCL 하이퍼파라미터
dcl_hyperparams = {
    'n': n,
    'N': N,
    'M': M,
    'H': H,
    'L': L,
    'w': w
}

# Lost Sales 환경 파라미터
env_params = {
    'lead_time': env_ls.tau,
    'holding_cost': env_ls.h,
    'penalty_cost': env_ls.p,
    'mean_demand': env_ls.mean_demand,       # 인스턴스 변수에서 가져옴
    'demand_std_dev': env_ls.demand_std_dev, # 인스턴스 변수에서 가져옴
    'max_action': env_ls.num_actions - 1
}

# --- 2. 모델 가중치와 파라미터들을 하나의 '체크포인트' 딕셔너리로 묶기 ---
checkpoint = {
    'model_state_dict': best_policy_lost_sales.state_dict(),
    'dcl_hyperparameters': dcl_hyperparams,
    'env_parameters': env_params
}

# --- 3. 파일 이름 생성 및 체크포인트 저장 ---
now = datetime.datetime.now()
timestamp = now.strftime("%Y%m%d_%H%M")
filename = f"dcl_lost_sales_checkpoint_{timestamp}.pth"

torch.save(checkpoint, filename)

print(f"모델과 하이퍼파라미터가 '{filename}' 파일로 성공적으로 저장되었습니다!")

모델과 하이퍼파라미터가 'dcl_lost_sales_checkpoint_20251012_0930.pth' 파일로 성공적으로 저장되었습니다!


In [24]:
# 셀 2: Perishable Inventory 문제

from dcl_definitions_final import (
    PerishableInventoryEnv, 
    BaseStockPolicy, 
    DCLTrainer,
    collect_samples_for_thread
)


if __name__ == '__main__':
    # --- 1. 문제 및 하이퍼파라미터 정의 ---
    print("\n" + "="*50)
    print("Perishable Inventory 문제에 대한 DCL 학습을 시작합니다.")
    print("="*50)

    # 변수 이름이 겹치지 않도록 '_pi' 접미사를 사용합니다.
    env_pi = PerishableInventoryEnv(
        lifetime=3,
        lead_time=2,
        holding_cost=500,
        penalty_cost=2000,
        waste_cost=2000,
        mean_demand=150,    # 전체 요일의 평균 수요는 15
        demand_std_dev=50,  # 요일별 평균 수요의 표준편차는 5
        max_action=300,
        fifo_ratio=0.4
    )
    initial_policy_pi = BaseStockPolicy(base_stock_level=400)#초기정책인 BSP를 400으로 설정

    # DCL 하이퍼파라미터
    n = 2     # 정책 반복 횟수
    N = 300   # 총 데이터 샘플 수, 오래걸려서 500->300으로 낮춤
    M = 20   # 상태-행동 쌍 당 외생 시나리오 수, 총 시뮬레이션 예산(Bₛ)을 결정하는 값, 오래걸려서 100에서 50으로 낮춤
    H = 20    # 롤아웃 깊이
    L = 20    # 웜업 기간, 초기값 50->20으로 낮춤
    w = 10    # 병렬 워커(프로세스) 수, w는 작업 프로세스에서 CPU 논리프로세스 수에 따라서

    # --- 2. DCL 트레이너 생성 및 학습 실행 ---
    dcl_trainer_pi = DCLTrainer(env_pi, initial_policy_pi, n, N, M, H, L, w)
    final_policies_pi = dcl_trainer_pi.train()

    # --- 3. 결과 ---
    best_policy_perishable = None
    if final_policies_pi:
        best_policy_perishable = final_policies_pi[-1]
        
        # <<< 테스트 상태에 요일 정보 추가 >>>
        # 예시 1: 금요일(인덱스 4)이고, 재고가 없는 상황
        inventory_part = np.zeros(env_pi.ml + max(env_pi.tau - 1, 0))
        day_part_fri = np.array([0, 0, 0, 0, 1, 0, 0])
        friday_state = np.concatenate((inventory_part, day_part_fri))
        
        action_fri = best_policy_perishable.get_action(friday_state)
        print(f"금요일에 재고가 없을 때 추천 주문량: {action_fri}")

        # 예시 2: 월요일(인덱스 0)이고, 재고가 없는 동일한 상황
        day_part_mon = np.array([1, 0, 0, 0, 0, 0, 0])
        monday_state = np.concatenate((inventory_part, day_part_mon))

        action_mon = best_policy_perishable.get_action(monday_state)
        print(f"월요일에 재고가 없을 때 추천 주문량: {action_mon}")


Perishable Inventory 문제에 대한 DCL 학습을 시작합니다.
Perishable: 자동 생성된 요일별 평균 수요:
  - 월: 157.59
  - 화: 95.98
  - 수: 177.76
  - 목: 139.11
  - 금: 83.57
  - 토: 224.60
  - 일: 158.38
DCL Trainer가 cuda 장치를 사용하여 학습을 시작합니다.

======= 시작: 근사 정책 반복 1/2 =======
데이터셋 K_0 생성을 시작합니다 (총 300개 샘플, 10개 워커)...

데이터셋 생성 완료. 총 300개 샘플 수집.
신경망 학습 시작...
Epoch 1/50, Validation Loss: 5.7659
Epoch 2/50, Validation Loss: 5.5017
Epoch 3/50, Validation Loss: 5.1159
Epoch 4/50, Validation Loss: 4.9334
Epoch 5/50, Validation Loss: 4.9411
Epoch 6/50, Validation Loss: 4.9606
Epoch 7/50, Validation Loss: 4.9641
Epoch 8/50, Validation Loss: 5.0130
Epoch 9/50, Validation Loss: 5.0219
조기 종료(Early stopping) 발동!
신경망 학습 완료.

======= 시작: 근사 정책 반복 2/2 =======
데이터셋 K_1 생성을 시작합니다 (총 300개 샘플, 10개 워커)...

데이터셋 생성 완료. 총 300개 샘플 수집.
신경망 학습 시작...
Epoch 1/50, Validation Loss: 4.4356
Epoch 2/50, Validation Loss: 4.3088
Epoch 3/50, Validation Loss: 4.3502
Epoch 4/50, Validation Loss: 4.1353
Epoch 5/50, Validation Loss: 4.0092
Epoch 6/50, Validat

In [28]:
# best_policy_perishable와 학습에 사용된 파라미터들이 있다고 가정

# --- 1. 저장하고 싶은 모든 파라미터를 딕셔너리로 정리 ---
# DCL 하이퍼파라미터 (이 부분은 Lost Sales와 동일)

from dcl_definitions_final import (
    PerishableInventoryEnv, 
    BaseStockPolicy, 
    DCLTrainer,
    collect_samples_for_thread
)



dcl_hyperparams = {
    'n': n,
    'N': N,
    'M': M,
    'H': H,
    'L': L,
    'w': w
}

# Perishable Inventory 환경 파라미터
# lifetime, waste_cost, fifo_ratio가 추가됩니다.
env_params_pi = {
    'lifetime': env_pi.ml,
    'lead_time': env_pi.tau,
    'holding_cost': env_pi.h,
    'penalty_cost': env_pi.p,
    'waste_cost': env_pi.w,
    """
    'mean_demand': env_pi.mean_demand,
    'demand_std_dev': env_pi.demand_std_dev,
    """
    'max_action': env_pi.num_actions - 1,
    'fifo_ratio': env_pi.fifo_ratio
}

# --- 2. 모델 가중치와 파라미터들을 하나의 '체크포인트' 딕셔너리로 묶기 ---
checkpoint_pi = {
    'model_state_dict': best_policy_perishable.state_dict(),
    'dcl_hyperparameters': dcl_hyperparams,
    'env_parameters': env_params_pi
}

# --- 3. 파일 이름 생성 및 체크포인트 저장 ---
now = datetime.datetime.now()
timestamp = now.strftime("%Y%m%d_%H%M")
filename_pi = f"dcl_perishable_checkpoint_{timestamp}.pth"

torch.save(checkpoint_pi, filename_pi)

print(f"모델과 하이퍼파라미터가 '{filename_pi}' 파일로 성공적으로 저장되었습니다!")

모델과 하이퍼파라미터가 'dcl_perishable_checkpoint_20251012_1448.pth' 파일로 성공적으로 저장되었습니다!


In [61]:
# 이 셀을 실행하기 전에 Perishable Inventory 문제에 대한 DCL 학습이 완료되어 'best_policy_perishable' 변수가 생성되어 있어야 합니다.


print("학습된 정책 모델에게 여러가지 재고 상황에 대해 질문합니다.")
print("-" * 50)

# --- 요일 정보 정의
# 요일은 월(0)부터 일(6)까지의 인덱스를 가집니다.
day_of_week_vector = np.array([0, 0, 0, 1, 0, 0, 0])

# --- 시나리오 1: 재고가 하나도 없는 상황 ---
# PerishableEnv의 상태 벡터는 [유통기한_ml, ..., 유통기한_1, 파이프라인_tau-1, ..., 파이프라인_1] + [요일]
state_1_inventory = np.array([0, 0, 0, 0])
state_1_full = np.concatenate((state_1_inventory, day_of_week_vector))
action_1 = best_policy_perishable.get_action(state_1_full)
print(f"상황 1: 재고가 전혀 없을 때  {state_1_full}")
print(f"▶ 추천 주문량: {action_1} 개")
print("-" * 50)


# --- 시나리오 2: 재고는 적지만, 곧 많이 들어올 예정인 상황 ---
state_2_inventory = np.array([0, 0, 100, 1000]) # 현재고 5, 파이프라인에 45개
state_2_full = np.concatenate((state_2_inventory, day_of_week_vector))
action_2 = best_policy_perishable.get_action(state_2_full)
print(f"상황 2: 재고는 적지만 파이프라인이 꽉 찼을 때 (월요일) {state_2_full}")
print(f"▶ 추천 주문량: {action_2} 개 ")
print("-" * 50)


# --- 시나리오 3: 재고가 매우 많은 상황 ---
state_3_inventory = np.array([100000, 1000, 1000, 10000])
state_3_full = np.concatenate((state_3_inventory, day_of_week_vector))
action_3 = best_policy_perishable.get_action(state_3_full)
print(f"상황 3: 재고가 매우 많을 때 (월요일) {state_3_full}")
print(f"▶ 추천 주문량: {action_3} 개 ")
print("-" * 50)

학습된 정책 모델에게 여러가지 재고 상황에 대해 질문합니다.
--------------------------------------------------
상황 1: 재고가 전혀 없을 때  [0 0 0 0 0 0 0 1 0 0 0]
▶ 추천 주문량: 127 개
--------------------------------------------------
상황 2: 재고는 적지만 파이프라인이 꽉 찼을 때 (월요일) [   0    0  100 1000    0    0    0    1    0    0    0]
▶ 추천 주문량: 75 개 
--------------------------------------------------
상황 3: 재고가 매우 많을 때 (월요일) [100000   1000   1000  10000      0      0      0      1      0      0
      0]
▶ 추천 주문량: 207 개 
--------------------------------------------------
